# Load EMOTIC and transform it into single-label

## 1. Data load

In [ ]:
import torchfile

data_train = torchfile.load("emotic_annotations/DiscreteContinuousAnnotations26_train.t7")
data_val   = torchfile.load("emotic_annotations/DiscreteContinuousAnnotations26_val.t7")
data_test  = torchfile.load("emotic_annotations/DiscreteContinuousAnnotations26_test.t7")

print(f"Train: {len(data_train)} rows")
print(f"Val: {len(data_val)} rows")
print(f"Test: {len(data_test)} rows")
print(f"Total: {len(data_train) + len(data_val) + len(data_test)} rows")

## 2. Structure

In [ ]:
print("Keys:", list(data_train[0].keys()))
print()
print("filename:", data_train[0][b"filename"])
print("workers:", data_train[0][b"workers"])
print("folder:", data_train[0][b"folder"])
print("head_bbox:", data_train[0][b"head_bbox"])
print("image_size:", data_train[0][b"image_size"])
print("personID:", data_train[0][b"personID"])
print("body_bbox:", data_train[0][b"body_bbox"])

## 3. Label map

In [ ]:
EMOTIC_LABELS = {
    1:  "Affection",
    2:  "Anger",
    3:  "Annoyance",
    4:  "Anticipation",
    5:  "Aversion",
    6:  "Confidence",
    7:  "Disapproval",
    8:  "Disconnection",
    9:  "Disquietment",
    10: "Doubt/Confusion",
    11: "Embarrassment",
    12: "Engagement",
    13: "Esteem",
    14: "Excitement",
    15: "Fatigue",
    16: "Fear",
    17: "Happiness",
    18: "Pain",
    19: "Peace",
    20: "Pleasure",
    21: "Sadness",
    22: "Sensitivity",
    23: "Suffering",
    24: "Surprise",
    25: "Sympathy",
    26: "Yearning",
}

## 4. Transform to DF

In [ ]:
import pandas as pd

def records_to_df(data, split_name):
    rows = []
    for rec in data:
        filename = rec[b"filename"].decode()
        folder = rec[b"folder"].decode()
        person_id = rec[b"personID"]
        body_bbox = rec[b"body_bbox"]
        head_bbox = rec[b"head_bbox"]
        img_size = rec[b"image_size"]
        for w in rec[b"workers"]:
            labels = [EMOTIC_LABELS[l] for l in w[b"labels"]]
            rows.append({
                "split": split_name,
                "filename": filename,
                "folder": folder,
                "person_id": person_id,
                "img_w": img_size[0],
                "img_h": img_size[1],
                "body_x1": body_bbox[0], "body_y1": body_bbox[1],
                "body_x2": body_bbox[2], "body_y2": body_bbox[3],
                "head_x1": head_bbox[0], "head_y1": head_bbox[1],
                "head_x2": head_bbox[2], "head_y2": head_bbox[3],
                "labels": labels,
            })
    return pd.DataFrame(rows)

df_train = records_to_df(data_train, "train")
df_val   = records_to_df(data_val, "val")
df_test  = records_to_df(data_test, "test")
df_all   = pd.concat([df_train, df_val, df_test], ignore_index=True)

print("Dims:", df_all.shape)
df_all.head()

## 5. Exploratory analysis

### Emotion freq


In [ ]:
## Frecuencia de cada emoción discreta
from collections import Counter

all_labels = [lbl for labels in df_all["labels"] for lbl in labels]
label_counts = pd.Series(Counter(all_labels)).sort_values(ascending=False)
print("Frecuencia de emociones")
print(label_counts.to_string())

### Basic emotions analysis

In [ ]:
basic_emotions = set(["Happiness", "Sadness", "Anger", "Fear", "Surprise", "Aversion"])

df_all["n_basic_emotions"] = df_all["labels"].apply(
    lambda lbls: len(set(lbls) & basic_emotions)
)

# Coocurrences
no_basics = (df_all["n_basic_emotions"] == 0).sum()
no_coocurrence   = (df_all["n_basic_emotions"] == 1).sum()
coocurrence = (df_all["n_basic_emotions"] > 1).sum()

print(f"Cases without basic emotions: {no_basics}")
print(f"Cases with coocurring basic emotions: {coocurrence}")
print(f"Cases without coocurring basic emotions: {no_coocurrence}")

from itertools import combinations

target_list = sorted(basic_emotions)
print("\nPair coocurrences:\n")
for a, b in combinations(target_list, 2):
    n = df_all["labels"].apply(lambda lbls: a in lbls and b in lbls).sum()
    print(f"  {a} + {b}: {n}")

## 6. Final DF with basic emotions alone

In [ ]:
df_final = df_all[df_all["n_basic_emotions"] == 1].copy()

df_final["basic_emotion"] = df_final["labels"].apply(
    lambda lbls: next(l for l in lbls if l in basic_emotions)
)

print(f"Original cases: {len(df_all)}")
print(f"Single basic emotion cases: {len(df_final)}")
print()
print("Basic emotion sdistribution:")
print(df_final["basic_emotion"].value_counts().to_string())

## 7. Save DF

In [ ]:
df_final.to_parquet("emotic_basic.parquet")